# ETF Pressure Lab — end-to-end research

This notebook is an interface to the tested Python modules. Run from a cloned repository after installing the locked environment. Data and outputs are not committed.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
from etf_pressure.pipeline import run
MODE = "cached"  # change to download for source refresh
summary = run(mode=MODE)
summary

Model decision_tree_price: validation Brier 0.2553


Model decision_tree_flow: validation Brier 0.2548


Model random_forest_price: validation Brier 0.2479


Model random_forest_flow: validation Brier 0.2475


Model gradient_boosting_price: validation Brier 0.2554


Model gradient_boosting_flow: validation Brier 0.2554


{'raw_rows': 58600,
 'feature_rows': 56080,
 'test_rows': 18240,
 'features': 49,
 'selected_model': 'random_forest',
 'last_date': '2026-11-02',
 'first_date': '2021-01-04',
 'created_utc': '2026-11-03T01:06:40.595620+00:00',
 'finra_coverage': 1.0,
 'paired_annualized_mean_difference_ci95': [-0.14765584574814386,
  -0.01326802566735481],
 'snapshot_sha256': 'd945c0780480a13d64e05f2b9f23116a490bdee4bb82a29bbe70f141062a6d22'}

In [2]:
import pandas as pd
pd.read_csv("artifacts/performance.csv")

,strategy,total_return,cagr,sharpe,max_drawdown,trades,mean_exposure,win_rate
0,FINRA + price,0.060882,0.032793,1.339004,-0.011648,5,0.010926,0.800000
1,Price only,0.194744,0.102020,1.578296,-0.027886,12,0.026221,0.916667
2,Buy the dip,-0.066545,-0.036899,-0.388787,-0.129989,51,0.110677,0.431373
3,SPY buy & hold,0.315374,0.161440,0.962868,-0.197662,1,1.000000,1.000000
4,ETF basket buy & hold,0.342003,0.174218,1.294303,-0.126802,1,1.000000,1.000000


In [3]:
import json
report = json.loads(Path("artifacts/models.json").read_text())
pd.DataFrame(report["experiments"]).drop(columns="tuning")

,name,variant,kind,depth,validation_brier,auc,brier,precision_at_055,observations
0,decision_tree_price,price,decision_tree,3,0.255305,0.538711,0.248383,0.464700,18240
1,decision_tree_flow,flow,decision_tree,3,0.254843,0.532274,0.248724,0.456416,18240
2,random_forest_price,price,random_forest,3,0.247869,0.557729,0.244599,0.741935,18240
3,random_forest_flow,flow,random_forest,3,0.247498,0.559494,0.244802,0.714286,18240
4,gradient_boosting_price,price,gradient_boosting,3,0.255444,0.563622,0.244827,0.636648,18240
5,gradient_boosting_flow,flow,gradient_boosting,3,0.255400,0.559548,0.245244,0.629827,18240


In [4]:
import plotly.express as px
curves = pd.read_parquet("artifacts/curves.parquet")
px.line(curves, x="date", y="equity", color="strategy")

In [5]:
pd.read_parquet("artifacts/latest.parquet")[["date", "ticker", "p_flow", "p_price", "selloff", "eligible"]].sort_values("p_flow", ascending=False)

,date,ticker,p_flow,p_price,selloff,eligible
21,2026-11-02,SLV,0.555156,0.553526,0,False
6,2026-11-02,EWZ,0.543818,0.555384,0,False
30,2026-11-02,XLC,0.538012,0.549701,0,False
8,2026-11-02,GDX,0.536756,0.544218,0,False
22,2026-11-02,SMH,0.536452,0.543928,1,False
34,2026-11-02,XLK,0.529920,0.537835,1,False
18,2026-11-02,QQQ,0.525205,0.532624,0,False
3,2026-11-02,EEM,0.516795,0.530426,0,False
9,2026-11-02,GLD,0.515511,0.507527,0,False
5,2026-11-02,EWJ,0.495598,0.497816,0,False


## Interpretation

Read docs/results.md for the measured outcome, costs, biases, and reproduction details. Launch the interactive monitor with `uv run streamlit run app.py`. Latest trades are research candidates, not brokerage instructions.